# 🛡️ Notebook 07: Uncertainty Calibration & Closed-Loop Safety Layer
**AI-Enabled Smart Hydroponic Nutrient Balancing System**  
*Sprint 7: Risk-Aware Actuation, MC Dropout Calibration & Hazard Prevention*

---

## 1. Executive Summary & Scientific Motivation

In physical agricultural automation, raw deep neural network predictions cannot be directly wired to chemical dosing pumps (nitric/phosphoric acid, potassium hydroxide, concentrated N-P-K salts). Even minor over-dosing causes irreversible root osmotic lysis or nutrient lockout.

To solve this, **Sprint 7** implements a dual-defense closed-loop architecture:
1. **Monte Carlo Dropout Uncertainty Estimation ($N=50$ stochastic passes):**
   - Computes predictive epistemic uncertainty $\sigma$ for every sensor prediction.
   - Gates are empirically calibrated on the **Validation Set** ($P_{75}$ for supervised alert, $P_{95}$ for automated abort).
2. **Deterministic Closed-Loop Safety Layer (`src/safety_layer.py`):**
   - **Tier 1 (Telemetry Integrity):** Traps sensor flatlines, ADC rail disconnections, and rate-of-change spikes.
   - **Tier 2 (Epistemic Risk Gate):** Blocks actuation if $\sigma > \text{HIGH\_THRESHOLD}$.
   - **Tier 3 (Biological Hazard Gate):** Inhibits massive automated dosing if conditions violate extreme biological limits.
   - **Tier 4 (Supervised Oversight):** Flags $\text{MED} < \sigma \le \text{HIGH}$ for human operator approval.
   - **Tier 5 (Safe Micro-Dosing):** Computes proportional dosing volume with single-cycle physical safety clamps.

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Ensure project root is in sys.path
ROOT_DIR = os.path.dirname(os.path.abspath(''))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

from src.safety_layer import make_dosing_decision, validate_sensor_reading, SafetyTriageEngine
print("Safety Layer & Uncertainty environment initialized.")

## 2. Load Calibrated Uncertainty Gates & Evaluation Records
Loading the structured evaluation logs generated by `notebooks/07_safety_evaluation.py`.

In [ ]:
exp_path = os.path.join(ROOT_DIR, 'experiments', 'exp_007_uncertainty_safety.json')
if os.path.exists(exp_path):
    with open(exp_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    print("### Validation-Calibrated Uncertainty Gates:")
    df_gates = pd.DataFrame(data['calibrated_thresholds']).T
    display(df_gates)
    
    print("\n### Fault Injection Prevention Results:")
    print(f"  - Total Faults Injected:    {data['fault_injection_test']['total_faults_injected']}")
    print(f"  - Hazardous Doses Prevented: {data['fault_injection_test']['bad_doses_prevented']}")
    print(f"  - Safety Prevention Rate:   {data['fault_injection_test']['prevention_rate_pct']}%")
else:
    print(f"File {exp_path} not found. Please run notebooks/07_safety_evaluation.py first.")

## 3. Publication Visualizations
Displaying reliability calibration diagrams, operational triage distributions, and error-uncertainty correlation scatter plots.

In [ ]:
fig_dir = os.path.join(ROOT_DIR, 'reports', 'figures')
fig_files = [
    ("Figure 21: Uncertainty Calibration & Reliability Diagram", "fig21_uncertainty_calibration.png"),
    ("Figure 22: Safety Layer Operational Triage & Fault Interception", "fig22_safety_layer_triage.png"),
    ("Figure 23: MC Dropout Uncertainty vs. Empirical Prediction Error", "fig23_mc_error_correlation.png"),
]

for title, fname in fig_files:
    fpath = os.path.join(fig_dir, fname)
    if os.path.exists(fpath):
        print(f"### {title}")
        display(Image(filename=fpath))
    else:
        print(f"Figure {fname} not found yet.")

## 4. Interactive Streaming Simulation Demo
Demonstrating real-time safety gating when a physical probe cable is severed mid-stream.

In [ ]:
# Example streaming simulation
calibrated_thresholds = {
    "pH": {"med_threshold": 0.050, "high_threshold": 0.100},
    "TDS": {"med_threshold": 20.0, "high_threshold": 45.0},
}
engine = SafetyTriageEngine(calibrated_thresholds=calibrated_thresholds)

# Step 1: Nominal condition with slight pH drift
d1 = engine.process_step(
    current_reading={"pH": 6.25, "TDS": 640.0, "DHT_temp": 22.0, "DHT_humidity": 60.0, "water_level": 2.0},
    prediction={"pH": 6.30, "TDS": 635.0, "water_level": 2.0},
    uncertainty={"pH": 0.025, "TDS": 12.0},
)
print(f"Step 1 (Slight Drift):  Action={d1.action:<12} TriageCode={d1.triage_code} Commands={d1.actuator_commands}")

# Step 2: High Epistemic Uncertainty injected
d2 = engine.process_step(
    current_reading={"pH": 6.25, "TDS": 640.0, "DHT_temp": 22.0, "DHT_humidity": 60.0, "water_level": 2.0},
    prediction={"pH": 6.45, "TDS": 600.0, "water_level": 2.0},
    uncertainty={"pH": 0.150, "TDS": 60.0},  # Exceeds HIGH_THRESHOLD
)
print(f"Step 2 (High Uncertainty): Action={d2.action:<12} TriageCode={d2.triage_code} Reason={d2.reason}")

# Step 3: Sensor Disconnection (Reading 0.0 TDS)
d3 = engine.process_step(
    current_reading={"pH": 6.25, "TDS": 0.0, "DHT_temp": 22.0, "DHT_humidity": 60.0, "water_level": 2.0},
    prediction={"pH": 6.25, "TDS": 0.0, "water_level": 2.0},
    uncertainty={"pH": 0.02, "TDS": 10.0},
)
print(f"Step 3 (Cable Disconnect): Action={d3.action:<12} TriageCode={d3.triage_code} Reason={d3.reason}")